# Asset Correlation Topology — Minimum Spanning Tree

**What this notebook does**  
Turns the historical return relationships of any Bloomberg universe (Index, Fund, Portfolio, ticker list or macro set) into an interactive Minimum Spanning Tree.  

You get an immediate visual answer to:
- Which assets form natural clusters?
- Which names are local hubs (most connected)?
- Which names act as *bridges* — the critical links that join otherwise separate groups?

**Why it is useful**  
A full correlation matrix is dense and hard to read. The MST keeps only the strongest non-redundant links (exactly $N-1$ edges for $N$ assets). The resulting network is easy to show in an IC memo, risk report or client presentation, and often reveals cross-sector relationships that a heatmap hides.

**How to use**  
1. Go to the **Setup** tab.  
2. Choose universe, date range, frequency (Daily / Weekly / Monthly), correlation method (Pearson or Spearman), minimum history % and minimum pairwise observations.  
3. Click **Run MST Analysis**.  
4. Explore the interactive chart and sector summary on the **Results** tab.  
5. Read the **About the MST** tab for the full methodology and interpretation notes.


**Design notes**  
- Node size = portfolio weight when a Portfolio is selected; otherwise equal.  
- “Largest Holding” in the sector table is only meaningful for portfolios.  
- This is a clean, transparent starting point. The highest-value next steps for portfolio managers are secondary/residual edges and risk-contribution overlays.

---

In [ ]:
import bql
import pandas as pd
import numpy as np
from scipy.sparse.csgraph import minimum_spanning_tree
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import bloomberg.bquant.portfolio as bqp
import custom_widgets

# ============================================================================
# BLOOMBERG CONNECTION
# ============================================================================
bq = bql.Service()

FREQUENCY_OPTIONS = ["Daily", "Weekly", "Monthly"]
CORR_METHOD_OPTIONS = ["Pearson", "Spearman"]

# ============================================================================
# UNIVERSE RESOLUTION
# ============================================================================
def resolve_universe(universe_picker):
    universe_type = universe_picker.universe_type
    selected_object = universe_picker._picker_type_dropdown.value

    if universe_type == "Portfolio":
        portfolio_id = selected_object._dropdown.value
        portfolio_label = selected_object._dropdown.label
        if not portfolio_id:
            raise ValueError("Please select a portfolio.")

        portfolio_df = (
            bqp.Portfolio
            .from_port(
                portfolio_id=portfolio_id,
                include_normalized_weights=True
            )
            .to_df()
            .reset_index()
        )
        if portfolio_df.empty:
            raise ValueError(
                f"No positions were returned for portfolio '{portfolio_label}'."
            )
        if "security" not in portfolio_df.columns:
            raise ValueError(
                "Portfolio data does not contain a 'security' column."
            )

        portfolio_df = portfolio_df[portfolio_df["security"].notna()].copy()
        securities = (
            portfolio_df["security"]
            .astype(str)
            .str.strip()
            .tolist()
        )

        weight_column = None
        for col in ["normalized_weight", "weight", "Weight", "Normalized Weight"]:
            if col in portfolio_df.columns:
                weight_column = col
                break

        if weight_column is None:
            print(
                "Warning: No portfolio weight column was found. "
                "Using equal weights for node sizing."
            )
            weights = pd.Series(1.0 / len(securities), index=securities)
        else:
            weight_series = pd.to_numeric(
                portfolio_df[weight_column], errors="coerce"
            )
            weights = pd.Series(weight_series.values, index=securities)
            weights = weights.replace([np.inf, -np.inf], np.nan).fillna(0.0)
            weights = weights.groupby(level=0).sum()
            weight_sum = weights.sum()
            if weight_sum > 0:
                weights = weights / weight_sum
            else:
                weights[:] = 1.0 / len(weights)

        securities = list(weights.index)
        return {
            "securities": securities,
            "weights": weights,
            "universe_type": universe_type,
            "universe_label": portfolio_label,
        }

    securities_or_universe = universe_picker.value
    if securities_or_universe is None:
        raise ValueError(
            f"No value has been supplied for the selected "
            f"universe type '{universe_type}'."
        )

    if isinstance(securities_or_universe, (list, tuple)):
        securities = [
            str(x).strip()
            for x in securities_or_universe
            if str(x).strip()
        ]
    else:
        securities = securities_or_universe

    return {
        "securities": securities,
        "weights": None,
        "universe_type": universe_type,
        "universe_label": universe_type,
    }

# ============================================================================
# DATA RETRIEVAL  (daily total returns – then converted if needed)
# ============================================================================
def get_universe_data(universe_picker, start_date, end_date):
    resolved = resolve_universe(universe_picker)
    securities = resolved["securities"]
    weights = resolved["weights"]
    universe_type = resolved["universe_type"]

    print(f"Universe Type: {universe_type}")

    req_returns = bql.Request(
        securities,
        {
            "Return": bq.data.day_to_day_total_return(
                dates=bq.func.range(start_date, end_date)
            )
        }
    )
    res_returns = bq.execute(req_returns)
    df_raw = res_returns[0].df().reset_index()

    if df_raw.empty:
        raise ValueError(
            "Bloomberg returned no return data for the selected universe."
        )

    date_col = "DATE" if "DATE" in df_raw.columns else None
    id_col = "ID" if "ID" in df_raw.columns else None
    return_col = "Return" if "Return" in df_raw.columns else None

    if not all([date_col, id_col, return_col]):
        raise ValueError(
            "Unexpected BQL return-data structure. "
            f"Columns returned: {list(df_raw.columns)}"
        )

    returns_df = df_raw.pivot(
        index=date_col, columns=id_col, values=return_col
    )
    returns_df.index = pd.to_datetime(returns_df.index, errors="coerce")
    returns_df = returns_df.sort_index()
    returns_df = returns_df.apply(pd.to_numeric, errors="coerce")

    # GICS sector
    req_sectors = bql.Request(
        securities, {"Sector": bq.data.gics_sector_name()}
    )
    res_sectors = bq.execute(req_sectors)
    meta_df = res_sectors[0].df()

    if meta_df.empty:
        meta_df = pd.DataFrame(index=returns_df.columns)
    if "ID" in meta_df.columns:
        meta_df = meta_df.set_index("ID")
    meta_df = meta_df.reindex(returns_df.columns)

    if "Sector" not in meta_df.columns:
        meta_df["Sector"] = "Unclassified"
    meta_df["Sector"] = (
        meta_df["Sector"].fillna("Unclassified").astype(str)
    )

    # Weights
    if weights is not None:
        weights.index = weights.index.astype(str).str.strip()
        weights = weights.reindex(returns_df.columns).fillna(0.0)
    else:
        n_assets = len(returns_df.columns)
        if n_assets > 0:
            weights = pd.Series(1.0 / n_assets, index=returns_df.columns)
        else:
            weights = pd.Series(dtype=float)

    meta_df["Weight"] = weights
    return returns_df, meta_df, resolved

# ============================================================================
# UNIT CHECK + FREQUENCY CONVERSION
# ============================================================================
def to_decimal_returns(returns_df):
    """
    Detect whether BQL returned percent (1.2) or decimal (0.012) values
    and normalise to decimal. Only needed for compounding; correlations
    themselves are scale-invariant.
    """
    values = returns_df.stack().abs()
    values = values[values > 0]
    if values.empty:
        return returns_df, "unknown"
    if values.median() > 0.1:
        return returns_df / 100.0, "percent (converted to decimal)"
    return returns_df, "decimal"

def convert_frequency(daily_returns, frequency):
    """
    Compound daily total returns into weekly (Friday close) or monthly
    (month-end) observations. Daily series is returned unchanged.
    """
    if frequency == "Daily":
        return daily_returns

    growth = 1.0 + daily_returns
    if frequency == "Weekly":
        grouped = growth.resample("W-FRI")
    elif frequency == "Monthly":
        try:
            grouped = growth.resample("ME")      # pandas >= 2.2
        except ValueError:
            grouped = growth.resample("M")       # older pandas
    else:
        raise ValueError(f"Unsupported frequency: {frequency}")

    return grouped.prod(min_count=1) - 1.0

# ============================================================================
# RETURN DATA CLEANING
# ============================================================================
def clean_returns(returns_df, min_history_pct=0.80):
    if returns_df.empty:
        raise ValueError("Return dataframe is empty.")

    returns_df = returns_df.copy()
    returns_df = returns_df.dropna(how="all")
    returns_df = returns_df.dropna(axis=1, how="all")

    if len(returns_df) < 12:
        raise ValueError(
            f"Only {len(returns_df)} observations are available at this "
            "frequency. Use a longer date range or a higher frequency."
        )
    if returns_df.shape[1] < 2:
        raise ValueError(
            "At least two assets with valid return data are required."
        )

    min_observations = int(np.ceil(len(returns_df) * min_history_pct))
    returns_df = returns_df.dropna(axis=1, thresh=min_observations)

    if returns_df.shape[1] < 2:
        raise ValueError(
            "Fewer than two assets remain after the "
            f"{min_history_pct:.0%} minimum-history filter."
        )
    return returns_df

# ============================================================================
# CORRELATION MATRIX
# ============================================================================
def calculate_correlation_matrix(returns_df, min_pairwise_observations, method="pearson"):
    method = method.lower()
    if method not in ("pearson", "spearman"):
        raise ValueError("Correlation method must be 'pearson' or 'spearman'.")

    corr_matrix = returns_df.corr(
        method=method,
        min_periods=min_pairwise_observations
    ).astype(float)

    # Iteratively drop the asset with the most missing pairwise correlations
    while corr_matrix.isna().any().any():
        worst = corr_matrix.isna().sum().idxmax()
        corr_matrix = corr_matrix.drop(index=worst, columns=worst)
        if corr_matrix.shape[0] < 2:
            raise ValueError(
                "Unable to construct a valid correlation matrix. "
                "Check the date range, frequency, min-history and "
                "min-pairwise settings."
            )

    corr_matrix = corr_matrix.clip(lower=-1.0, upper=1.0)
    corr_matrix = (corr_matrix + corr_matrix.T) / 2.0
    np.fill_diagonal(corr_matrix.values, 1.0)
    return corr_matrix

# ============================================================================
# MANTEGNA DISTANCE
# ============================================================================
def correlation_to_distance(corr_matrix):
    corr_values = np.clip(
        np.asarray(corr_matrix.values, dtype=float), -1.0, 1.0
    )
    distance_matrix = np.sqrt(np.maximum(0.0, 2.0 * (1.0 - corr_values)))
    np.fill_diagonal(distance_matrix, 0.0)
    distance_matrix = (distance_matrix + distance_matrix.T) / 2.0
    return distance_matrix

# ============================================================================
# NODE BETWEENNESS FOR A TREE
# ============================================================================
def calculate_tree_node_betweenness(adjacency):
    n = adjacency.shape[0]
    if n <= 2:
        return np.zeros(n)

    betweenness = np.zeros(n)
    for removed_node in range(n):
        visited = set()
        component_sizes = []
        for start in range(n):
            if start == removed_node or start in visited:
                continue
            stack = [start]
            visited.add(start)
            component_size = 0
            while stack:
                current = stack.pop()
                component_size += 1
                for neighbour in np.where(adjacency[current] > 0)[0]:
                    if neighbour == removed_node:
                        continue
                    if neighbour not in visited:
                        visited.add(neighbour)
                        stack.append(neighbour)
            component_sizes.append(component_size)

        total = 0
        running = 0
        for size in component_sizes:
            total += running * size
            running += size
        betweenness[removed_node] = total
    return betweenness

# ============================================================================
# MST CONSTRUCTION + METRICS
# ============================================================================
def compute_mst_and_metrics(corr_matrix, meta_df, universe_type):
    labels = corr_matrix.columns.tolist()
    n = len(labels)
    if n < 2:
        raise ValueError("At least two assets are required for an MST.")

    distance_matrix = correlation_to_distance(corr_matrix)
    mst = minimum_spanning_tree(distance_matrix)
    mst_dense = mst.toarray()

    adjacency = np.zeros((n, n), dtype=float)
    edges = []
    for i in range(n):
        for j in range(i + 1, n):
            d1 = mst_dense[i, j]
            d2 = mst_dense[j, i]
            if d1 > 0:
                distance = d1
            elif d2 > 0:
                distance = d2
            else:
                continue
            correlation = float(
                np.clip(1.0 - (distance ** 2) / 2.0, -1.0, 1.0)
            )
            adjacency[i, j] = 1
            adjacency[j, i] = 1
            edges.append(
                {
                    "source": labels[i],
                    "target": labels[j],
                    "distance": float(distance),
                    "correlation": correlation,
                }
            )

    if len(edges) != n - 1:
        raise ValueError(
            f"MST construction produced {len(edges)} edges "
            f"for {n} assets. Expected {n - 1}."
        )

    degrees = adjacency.sum(axis=0)
    most_connected_idx = int(np.argmax(degrees))
    most_connected_stock = labels[most_connected_idx]

    node_betweenness = calculate_tree_node_betweenness(adjacency)
    bridge_idx = int(np.argmax(node_betweenness))
    bridge_stock = labels[bridge_idx]

    meta_df = meta_df.reindex(labels).copy()
    meta_df["Sector"] = meta_df["Sector"].fillna("Unclassified").astype(str)
    meta_df["Weight"] = pd.to_numeric(
        meta_df["Weight"], errors="coerce"
    ).fillna(0.0)

    pairwise = corr_matrix.values[np.triu_indices(n, k=1)]
    avg_corr = float(np.mean(pairwise)) if len(pairwise) else np.nan

    # Sector / Cluster summary
    sector_summary = []
    is_portfolio = (universe_type == "Portfolio")

    for sector, grp in meta_df.groupby("Sector", sort=True):
        sector_assets = grp.index.tolist()
        if len(sector_assets) > 1:
            sub_corr = corr_matrix.loc[sector_assets, sector_assets]
            sub_values = sub_corr.values[
                np.triu_indices(len(sector_assets), k=1)
            ]
            avg_sub_corr = float(np.mean(sub_values))
            avg_text = f"{avg_sub_corr:.2f}"
        else:
            avg_text = "N/A"

        if is_portfolio:
            largest_holding = str(grp["Weight"].idxmax()).split()[0]
        else:
            largest_holding = "N/A (equal weights)"

        sector_summary.append(
            {
                "Cluster/Sector": sector,
                "Assets": len(grp),
                "Weight": f"{grp['Weight'].sum() * 100:.1f}%",
                "Largest Holding": largest_holding,
                "Avg Corr": avg_text,
            }
        )

    cluster_table = pd.DataFrame(sector_summary)

    kpis = {
        "avg_corr": f"{avg_corr:.2f}" if not np.isnan(avg_corr) else "N/A",
        "most_connected": str(most_connected_stock).split()[0],
        "most_connected_degree": int(degrees[most_connected_idx]),
        "bridge_stock": str(bridge_stock).split()[0],
        "bridge_betweenness": int(node_betweenness[bridge_idx]),
        "bridge_normalized": float(
            node_betweenness[bridge_idx] / ((n - 1) * (n - 2) / 2)
        ) if n > 2 else 0.0,
    }

    return edges, labels, kpis, cluster_table, meta_df, node_betweenness

# ============================================================================
# LAYOUT (attraction only along MST edges)
# ============================================================================
def weighted_fruchterman_reingold(edges, labels, iterations=80):
    n = len(labels)
    if n == 0:
        return {}

    rng = np.random.default_rng(42)
    pos = {label: rng.random(2) for label in labels}
    k = np.sqrt(1.0 / max(n, 1))
    temperature = 0.1
    dt = temperature / (iterations + 1)

    for _ in range(iterations):
        displacement = {label: np.zeros(2) for label in labels}

        # Repulsion
        for i, v in enumerate(labels):
            for u in labels[i + 1:]:
                delta = pos[v] - pos[u]
                spatial_distance = np.linalg.norm(delta)
                if spatial_distance <= 1e-12:
                    continue
                force = (k ** 2) / spatial_distance
                direction = delta / spatial_distance
                displacement[v] += direction * force
                displacement[u] -= direction * force

        # Attraction along MST edges only
        for edge in edges:
            u = edge["source"]
            v = edge["target"]
            m_distance = max(edge["distance"], 1e-6)
            delta = pos[v] - pos[u]
            spatial_distance = np.linalg.norm(delta)
            if spatial_distance <= 1e-12:
                continue
            force = (spatial_distance ** 2) / (k * (m_distance ** 1.5))
            direction = delta / spatial_distance
            displacement[v] -= direction * force
            displacement[u] += direction * force

        for label in labels:
            disp_len = np.linalg.norm(displacement[label])
            if disp_len <= 1e-12:
                continue
            move = min(disp_len, temperature)
            pos[label] += (displacement[label] / disp_len) * move

        temperature -= dt

    return pos

# ============================================================================
# VISUALISATION
# ============================================================================
def visualize_mst(
    edges, labels, pos, meta_df, node_betweenness, universe_type, frequency
):
    sectors = meta_df["Sector"].fillna("Unclassified").unique().tolist()
    color_palette = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    ]
    sector_color_map = {
        sector: color_palette[i % len(color_palette)]
        for i, sector in enumerate(sectors)
    }

    edge_traces = []
    for edge in edges:
        x0, y0 = pos[edge["source"]]
        x1, y1 = pos[edge["target"]]
        corr = edge["correlation"]
        opacity = max(0.15, min(1.0, (corr + 1.0) / 2.0))
        width = 1.0 + max(corr, 0) * 4.0
        edge_color = (
            f"rgba(0,128,0,{opacity})"
            if corr > 0.4
            else f"rgba(235,179,52,{opacity})"
        )
        edge_traces.append(
            go.Scatter(
                x=[x0, x1, None],
                y=[y0, y1, None],
                mode="lines",
                line=dict(color=edge_color, width=width),
                hoverinfo="text",
                text=(
                    f"{edge['source']} – {edge['target']}<br>"
                    f"Correlation: {corr:.2f}<br>"
                    f"Mantegna Distance: {edge['distance']:.3f}"
                ),
                showlegend=False
            )
        )

    node_x, node_y, node_sizes = [], [], []
    node_colors, node_labels, hover_text = [], [], []

    for idx, label in enumerate(labels):
        x, y = pos[label]
        node_x.append(x)
        node_y.append(y)
        weight = float(meta_df.loc[label, "Weight"])
        node_sizes.append(max(10, min(45, 10 + weight * 350)))
        sector = meta_df.loc[label, "Sector"]
        node_colors.append(sector_color_map.get(sector, "#7f7f7f"))
        node_labels.append(str(label).split()[0])
        hover_text.append(
            f"Asset: {label}<br>"
            f"Sector: {sector}<br>"
            f"Weight: {weight * 100:.2f}%<br>"
            f"Node Betweenness: {node_betweenness[idx]:.0f}"
        )

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        marker=dict(
            size=node_sizes,
            color=node_colors,
            line=dict(width=1.5, color="white")
        ),
        text=node_labels,
        textposition="top center",
        textfont=dict(size=10, color="#2c3e50"),
        hoverinfo="text",
        hovertext=hover_text,
        showlegend=False
    )

    fig = go.Figure(data=edge_traces + [node_trace])
    fig.update_layout(
        title=dict(
            text=(
                "Asset Correlation Topology — Minimum Spanning Tree "
                f"({universe_type}, {frequency})"
            ),
            x=0.5,
            font=dict(size=16, color="#2c3e50")
        ),
        hovermode="closest",
        margin=dict(b=10, l=10, r=10, t=60),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        plot_bgcolor="white",
        width=1100,
        height=680
    )
    return fig

# ============================================================================
# FULL MST PIPELINE
# ============================================================================
def run_mst_analysis(
    universe_picker,
    start_date,
    end_date,
    frequency,
    min_history_pct,
    corr_method,
    min_pairwise
):
    returns_df, meta_df, resolved = get_universe_data(
        universe_picker, start_date, end_date
    )
    print(f"Initial assets returned: {returns_df.shape[1]}")

    returns_df, units = to_decimal_returns(returns_df)
    print(f"Daily return units detected: {units}")

    returns_df = convert_frequency(returns_df, frequency)
    print(f"{frequency} observations (before cleaning): {len(returns_df)}")

    returns_df = clean_returns(returns_df, min_history_pct=min_history_pct)
    print(f"Assets after history filter: {returns_df.shape[1]}")

    n_obs = len(returns_df)
    # Use the user-supplied minimum, but never exceed the available observations
    effective_min_pairwise = min(min_pairwise, n_obs)
    if effective_min_pairwise < 10:
        effective_min_pairwise = 10

    corr_matrix = calculate_correlation_matrix(
        returns_df,
        min_pairwise_observations=effective_min_pairwise,
        method=corr_method
    )
    print(
        f"Assets in correlation matrix: {corr_matrix.shape[0]} "
        f"(method={corr_method}, min pairwise observations: {effective_min_pairwise})"
    )

    meta_df = meta_df.reindex(corr_matrix.columns)
    weight_sum = meta_df["Weight"].sum()
    if weight_sum > 0:
        meta_df["Weight"] = meta_df["Weight"] / weight_sum
    else:
        meta_df["Weight"] = 1.0 / len(meta_df)

    edges, labels, kpis, cluster_table, meta_df, node_betweenness = (
        compute_mst_and_metrics(
            corr_matrix, meta_df, resolved["universe_type"]
        )
    )

    pos = weighted_fruchterman_reingold(edges, labels)
    fig = visualize_mst(
        edges, labels, pos, meta_df, node_betweenness,
        resolved["universe_type"], frequency
    )

    resolved["frequency"] = frequency
    resolved["n_obs"] = n_obs
    resolved["corr_method"] = corr_method
    resolved["min_pairwise"] = effective_min_pairwise

    return (
        fig, kpis, cluster_table, corr_matrix, meta_df, edges, resolved
    )

# ============================================================================
# USER INTERFACE
# ============================================================================
UNIVERSE_TYPES = [
    "index",
    "fund",
    "ticker list",
    "portfolio",
    "global macro movers",
]

universe_picker = custom_widgets.UniversePicker(
    bq,
    univ_types=UNIVERSE_TYPES,
    value="B500 Index",
    description="Universe Type",
    layout={"min_width": "700px"}
)

start_date_input = widgets.DatePicker(
    description="Start Date:",
    value=pd.Timestamp("2023-01-01"),
    style={"description_width": "130px"}
)

end_date_input = widgets.DatePicker(
    description="End Date:",
    value=(pd.Timestamp.now() - pd.Timedelta(days=1)),
    style={"description_width": "130px"}
)

frequency_dropdown = widgets.Dropdown(
    options=FREQUENCY_OPTIONS,
    value="Daily",
    description="Frequency:",
    style={"description_width": "130px"},
    layout={"width": "280px"}
)

corr_method_dropdown = widgets.Dropdown(
    options=CORR_METHOD_OPTIONS,
    value="Pearson",
    description="Corr. method:",
    style={"description_width": "130px"},
    layout={"width": "280px"}
)

min_history_slider = widgets.IntSlider(
    value=80,
    min=50,
    max=100,
    step=5,
    description="Min history (%):",
    style={"description_width": "130px"},
    layout={"width": "450px"}
)

min_pairwise_slider = widgets.IntSlider(
    value=60,
    min=10,
    max=252,
    step=5,
    description="Min pairwise obs:",
    style={"description_width": "130px"},
    layout={"width": "450px"}
)

setting_help = widgets.HTML(
    """
    <div style="font-family:sans-serif;font-size:12px;color:#555;
                max-width:900px;padding:8px 0 12px 0;line-height:1.55;">
        <b>Frequency</b> — the return interval used for the correlation matrix.
        Weekly and monthly series are compounded from daily <i>total</i> returns
        (dividends included). Daily data can understate cross-market correlation
        because of non-synchronous closing times; weekly data reduces this bias.
        Monthly data needs a long history (preferably 5+ years) because the number
        of observations becomes small.<br><br>

        <b>Correlation method</b><br>
        • <b>Pearson</b> (default) measures linear co-movement. Sensitive to
          outliers and extreme returns.<br>
        • <b>Spearman</b> measures rank (monotonic) co-movement. More robust to
          outliers and non-linear relationships. Often produces slightly different
          hubs and bridges when a few large moves dominate the sample.<br><br>

        <b>Min history (%)</b> — an asset is kept only if it has valid returns for
        at least this percentage of the periods <i>at the chosen frequency</i>
        (default 80%). Lowering it admits short-history names whose correlations
        are noisier and can distort the tree.<br><br>

        <b>Min pairwise observations</b> — the minimum number of overlapping
        non-NaN observations required for any single pairwise correlation to be
        calculated (default 60). Pairs that fall short are treated as missing and
        the worst offenders are iteratively dropped until a complete matrix remains.
        Raising this value makes the matrix stricter and can remove more assets.
    </div>
    """
)

run_button = widgets.Button(
    description="Run MST Analysis",
    button_style="success",
    layout={"width": "200px"}
)

output = widgets.Output()

# ============================================================================
# RUN CALLBACK
# ============================================================================
def on_run_click(_):
    ui_display.selected_index = 1
    with output:
        output.clear_output(wait=True)
        try:
            if start_date_input.value is None or end_date_input.value is None:
                raise ValueError("Please select both a start date and end date.")
            if start_date_input.value >= end_date_input.value:
                raise ValueError("Start date must be earlier than end date.")

            start_date = start_date_input.value.strftime("%Y-%m-%d")
            end_date = end_date_input.value.strftime("%Y-%m-%d")
            frequency = frequency_dropdown.value
            min_history_pct = min_history_slider.value / 100.0
            corr_method = corr_method_dropdown.value
            min_pairwise = min_pairwise_slider.value

            print("Processing correlation topology...")
            (
                fig, kpis, cluster_table, corr_matrix,
                meta_df, edges, resolved
            ) = run_mst_analysis(
                universe_picker, start_date, end_date,
                frequency, min_history_pct, corr_method, min_pairwise
            )

            def kpi_box(title, value, colour="#2c3e50", size=22):
                return f"""
                <div style="text-align:center;">
                    <span style="color:#6c757d;font-size:11px;
                                 text-transform:uppercase;font-weight:bold;">
                        {title}
                    </span><br>
                    <b style="font-size:{size}px;color:{colour};">{value}</b>
                </div>
                """

            kpi_html = f"""
            <div style="display:flex;justify-content:space-around;
                        background-color:#f8f9fa;padding:15px;
                        border-radius:8px;margin-bottom:10px;
                        border:1px solid #e9ecef;font-family:sans-serif;">
                {kpi_box("Universe", resolved["universe_type"], size=18)}
                {kpi_box("Frequency", resolved["frequency"], size=18)}
                {kpi_box("Method", resolved["corr_method"], size=18)}
                {kpi_box("Observations", resolved["n_obs"])}
                {kpi_box("Assets", len(corr_matrix))}
                {kpi_box("Avg Correlation", kpis["avg_corr"])}
                {kpi_box("Most Connected", kpis["most_connected"], "#e74c3c")}
                {kpi_box("Bridge Asset", kpis["bridge_stock"], "#3498db")}
            </div>
            """
            display(widgets.HTML(kpi_html))
            fig.show()

            display(
                widgets.HTML(
                    """
                    <h4 style="font-family:sans-serif;margin-top:20px;
                               margin-bottom:5px;color:#2c3e50;">
                        Cluster / Sector Summary
                    </h4>
                    <p style="font-family:sans-serif;font-size:12px;color:#666;
                              margin-top:0;">
                        “Largest Holding” shows the highest portfolio weight inside
                        each sector when a Portfolio is selected. For indices,
                        funds and ticker lists the weights are equal, so the column
                        displays “N/A (equal weights)”.
                    </p>
                    """
                )
            )
            display(cluster_table)

            print(
                f"\nMST contains {len(edges)} edges "
                f"connecting {len(corr_matrix)} assets."
            )
            print(
                f"Most connected asset: {kpis['most_connected']} "
                f"({kpis['most_connected_degree']} connections)"
            )
            print(
                f"Bridge asset: {kpis['bridge_stock']} "
                f"(node betweenness {kpis['bridge_betweenness']}, "
                f"normalised {kpis['bridge_normalized']:.2f})"
            )

        except Exception as e:
            print(f"Execution Error: {e}")
            import traceback
            traceback.print_exc()

run_button.on_click(on_run_click)

# ============================================================================
# SETUP TAB
# ============================================================================
setup_description = widgets.HTML(
    """
    <div style="font-family:sans-serif;max-width:900px;padding:5px 0 15px 0;">
        <h3 style="color:#2c3e50;margin-bottom:6px;">MST Analysis Setup</h3>
        <p style="color:#555;margin-top:0;">
            Select the universe, historical window, return frequency and
            correlation settings. The Minimum Spanning Tree is built from the
            resulting correlation matrix using Mantegna distance.
            Portfolio weights affect node size and the “Largest Holding” column;
            they do not affect the tree structure itself.
        </p>
    </div>
    """
)

inputs = widgets.VBox(
    [
        setup_description,
        universe_picker,
        start_date_input,
        end_date_input,
        frequency_dropdown,
        corr_method_dropdown,
        min_history_slider,
        min_pairwise_slider,
        setting_help,
        run_button,
    ],
    layout={"padding": "10px"}
)

# ============================================================================
# ABOUT TAB
# ============================================================================
about_html = widgets.HTML(
    """
    <div style="font-family:sans-serif;max-width:920px;background:#f8f9fa;
                border:1px solid #e9ecef;border-radius:8px;padding:18px 24px;
                color:#2c3e50;font-size:13px;line-height:1.6;">

    <h3 style="margin-top:0;">What this analysis does</h3>
    <p>
        The Minimum Spanning Tree (MST) converts the historical return
        relationships between assets into a network. Each asset is a
        <b>node</b>; the strongest relationships become <b>edges</b>.
    </p>
    <p>
        Pairwise correlations are calculated, then converted to Mantegna
        distance:<br>
        <code>d = √(2 × (1 − ρ))</code><br>
        Highly correlated assets are close together; weakly or negatively
        correlated assets are far apart. The MST retains the smallest set of
        links that still connects every asset — for <b>N assets there are
        always exactly N − 1 edges</b>.
    </p>

    <h4>Why use an MST?</h4>
    <p>
        A full correlation matrix contains thousands of numbers. The MST
        removes redundancy and keeps the structural skeleton. It makes
        clusters, hub assets, unusual cross-sector links and bridge assets
        immediately visible.
    </p>

    <h4>Most Connected vs Bridge Asset</h4>
    <p>
        <b>Most Connected</b> is the asset with the highest number of direct
        links in the tree (a local hub).
    </p>
    <p>
        <b>Bridge Asset</b> is the asset with the highest
        <b>node betweenness</b> — the one that lies on the largest number of
        shortest paths between other pairs of assets. Even if it has only a
        few direct connections, removing it would split the network into
        disconnected groups. High betweenness describes structural position,
        not size, volatility or average correlation.
    </p>

    <h4>How to read the chart</h4>
    <p>
        <b>Node colour</b> = GICS sector.<br>
        <b>Node size</b> = portfolio weight (equal size for non-portfolio
        universes).<br>
        <b>Green, thicker edges</b> = correlation &gt; 0.4;<br>
        <b>Amber edges</b> = weaker links.<br>
        Absolute positions on the page have no meaning; only the connections
        and relative closeness matter. Hover for exact values.
    </p>

    <h4>Observation frequency</h4>
    <p>
        Correlations can be measured on <b>Daily, Weekly or Monthly</b>
        total returns. Weekly and monthly series are compounded from daily
        total returns (dividends included). This approach keeps a consistent
        total-return definition across frequencies.
    </p>
    <p>
        Daily data across multiple markets is affected by non-synchronous
        closing times, which can understate cross-market correlation. Weekly
        data mitigates this. Monthly data yields far fewer observations, so
        a long date range is essential.
    </p>

    <h4>Minimum history filter</h4>
    <p>
        After the returns have been converted to the chosen frequency, an
        asset is retained only if it has valid (non-NaN) observations for at
        least the selected percentage of those periods (default 80%).
        Assets that fail the filter are dropped before the correlation matrix
        is built. This is why the headline “Observations” number can be lower
        than the “observations before cleaning” print-out.
    </p>

    <h4>Minimum pairwise observations</h4>
    <p>
        Even after the history filter, some pairs of assets may still have
        insufficient overlapping observations. The correlation matrix requires
        at least this many overlapping points for a correlation to be
        calculated (default 60). Pairs that fall short are treated as missing;
        the algorithm then iteratively removes the worst offenders until a
        complete matrix remains.
    </p>

    <h4>Pearson vs Spearman</h4>
    <p>
        <b>Pearson</b> measures linear co-movement and is the classic choice.
        It is sensitive to large outliers.<br>
        <b>Spearman</b> measures rank (monotonic) co-movement and is more
        robust to extreme returns. Switching method can change which assets
        appear as hubs or bridges, especially in periods that contain a few
        very large moves.
    </p>

    <h4>Cluster / Sector Summary</h4>
    <p>
        “Largest Holding” reports the highest portfolio weight inside each
        sector when a Portfolio is selected. For indices, funds and ticker
        lists the analysis uses equal weights, so the column shows
        “N/A (equal weights)”.
    </p>

    <h4>Average correlation</h4>
    <p>
        The simple mean of all unique pairwise correlations after filtering.
        For a broad equity universe on daily data, values in the 0.15–0.35
        range are typical. Weekly and monthly correlations are usually higher
        because noise and non-synchronicity are reduced.
    </p>
    </div>
    """
)

# ============================================================================
# TABS
# ============================================================================
ui_display = widgets.Tab(
    children=[inputs, output, about_html],
    layout={"min_height": "780px"}
)
ui_display.set_title(0, "Setup")
ui_display.set_title(1, "Results Dashboard")
ui_display.set_title(2, "About the MST")

display(ui_display)